In [4]:
%load_ext autoreload
%autoreload 2

In [89]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
print("PROJECT ROOT ==>", PROJECT_ROOT)

sys.path.append(str(PROJECT_ROOT))
from src.mesh_fun import (
    load_segmentation,load_pred_landmarks,visualize_pred_and_gt,visualize,load_mesh,
    color_mesh_by_groups,create_landmark_spheres,load_landmarks,
    load_gt_landmarks,save_screenshot
)
from src.utils_fun import build_perturbation_label
from src.landmark_eval import( 
    compute_all_statistics,
    evaluate_single_scan_from_landmarks,
    quality_profile_for_scan,
select_examples_by_quantile, 
compute_profiles_for_scans, 
build_synthetic_oracle_dataset, 
build_missing_landmark_configs,
build_class_flip_configs, 
build_landmark_shift_configs
)
from src.var_constants import CATEGORIES,TOOTH_GROUP_PALETTE,LANDMARK_PALETTE,TOOTH_TO_GROUP


PROJECT ROOT ==> C:\Users\aless\Tesi_Msc\multi_agent_evaluation_on_intraoral_3D_scans


In [7]:
#PROJECT_ROOT = Path(__file__).resolve().parent
DATA_ROOT = PROJECT_ROOT / "dataset"
SCANS = DATA_ROOT / "toothinstancenet_input"
GT_ROOT = SCANS
PRED_CSV = DATA_ROOT / "model_predictions" / "ynlab" / "predictions.csv"
SCREENSHOT_DIR = DATA_ROOT / "screenshot_scans" / "raw"

In [8]:
scan_name='AKBDPB4C_upper'
scan_name='RZ69JRI9_lower'

In [9]:
screenshot_mesh_path =SCREENSHOT_DIR / scan_name
mesh_path = SCANS / f"{scan_name}.obj"
gt_json = GT_ROOT / f"{scan_name}__kpt.json"
seg_path  =SCANS / f"{scan_name}_seg.json"
tooth_seg_labels = load_segmentation(seg_path)

#### funzioni di pertbuazione

In [90]:
import random
import numpy as np
#Rimuove una percentuale di landmark
def apply_missing_landmark_perturbation(
    coords,
    classes,
    fraction,
    #specifica opzionalmente un sotto-insieme di classi da considerare. Es. Mesial
    target_landmark_classes=[],
    seed=42,
):
    rng = random.Random(seed)
    if len(coords) == 0:
        return coords.copy(), classes.copy()
    if len(target_landmark_classes)==0:
        candidate_idx = list(range(len(coords)))
    else:
        candidate_idx = [i for i, cls in enumerate(classes) if cls in target_landmark_classes]
    if len(candidate_idx) == 0:
        return ( coords.copy(),classes.copy() )
    n_remove = max( 1, int(len(candidate_idx) * fraction))

    remove_idx = set(rng.sample(  candidate_idx, min(    n_remove,len(candidate_idx))) )
    keep_idx = [i for i in range(len(coords)) if i not in remove_idx]
    coords_new = coords[keep_idx]
    classes_new = [classes[i] for i in keep_idx]
    return (
        coords_new,
        classes_new,
    )

def apply_class_flip_perturbation(
    coords,
    classes,
    fraction,
    target_landmark_classes=[],
    seed=42,
):
    MESH_CLASSES = list(set(classes))
    rng = random.Random(seed)
    if len(classes) == 0:
        return coords.copy(), classes.copy()

    #GLOBAL FLIP==> considera tutti i punti
    if len(target_landmark_classes) == 0:
        candidate_idx = list(range(len(classes)))
        print('### global flip with len(candidate)==>',len(candidate_idx))
    #LOCALIZED FLIP=> solo specifica classe target
    else:
        candidate_idx = [ i for i, cls in enumerate(classes) if cls in target_landmark_classes]

    if len(candidate_idx) == 0:
        return (coords.copy(),classes.copy())
    n_flip = max(1,int(len(candidate_idx) * fraction))
    print('## num flip==>',n_flip)
    flip_idx = rng.sample(candidate_idx,min(n_flip, len(candidate_idx))
                          )
    classes_new = classes.copy()
    for idx in flip_idx:
        current_class = classes_new[idx]
        candidate_classes = [ c for c in MESH_CLASSES if c != current_class]
        if len(candidate_classes) == 0:
            continue
        new_class = rng.choice(candidate_classes)
        classes_new[idx] = new_class

    return (coords.copy(),classes_new)

def apply_landmark_shift_perturbation(
    coords,
    classes,
    fraction,
    shift_mm,
    target_landmark_classes=[],
    seed=42,
):
    rng = random.Random(seed)
    if len(coords) == 0:
        return coords.copy(), classes.copy()

    #effetto GLOBAL 
    if len(target_landmark_classes) == 0:
        candidate_idx = list(range(len(coords)))
        print("### global shift with len(candidate)==>",len(candidate_idx))
    #LOCALIZED SHIFT
    else:
        candidate_idx = [i for i, cls in enumerate(classes) if cls in target_landmark_classes
        ]
    if len(candidate_idx) == 0:
        return (
            coords.copy(),
            classes.copy(),
        )
    n_shift = max(1,int(len(candidate_idx) * fraction))
    print("## num shift==>", n_shift)
    shift_idx = rng.sample(
        candidate_idx,
        min(n_shift, len(candidate_idx))
    )
    coords_new = coords.copy()
    rng_np = np.random.default_rng(seed)
    for idx in shift_idx:

        direction = rng_np.normal(0,1,size=3)

        norm = np.linalg.norm(direction)
        if norm < 1e-8:
            continue
        direction = direction / norm
        #modulo pari a  shift_mm
        magnitude = shift_mm
        coords_new[idx] = (coords_new[idx]+ direction * magnitude)
    return (
        coords_new,
        classes.copy(),
    )

In [91]:
coords_pred, classes_pred = load_pred_landmarks(PRED_CSV, scan_name)
coords_gt, classes_gt = load_gt_landmarks(gt_json)

noisy_coords, noisy_classes=apply_missing_landmark_perturbation(
    coords_pred,
    classes_pred,
    fraction=0.990,
    target_landmark_classes='Mesial',
    seed=42)
flip_coords, flip_classes = apply_class_flip_perturbation(
    coords_pred,
    classes_pred,
    fraction=0.2,
    target_landmark_classes=['Mesial'],
    seed=42,
)
shift_coords, shift_classes = apply_landmark_shift_perturbation(
    coords_pred,
    classes_pred,
    fraction=0.6,
    target_landmark_classes=[],
    shift_mm=0.5,
    seed=42,
)
mesh = load_mesh(mesh_path)
mesh = color_mesh_by_groups(mesh, tooth_seg_labels, TOOTH_TO_GROUP, TOOTH_GROUP_PALETTE)
spheres_pred = create_landmark_spheres(coords_pred, classes_pred, LANDMARK_PALETTE)
spheres_gt = create_landmark_spheres(coords_gt, classes_gt, LANDMARK_PALETTE)

spheres_noisy = create_landmark_spheres(shift_coords, shift_classes, LANDMARK_PALETTE)

#visualize(mesh, spheres_pred, view="front")
#visualize(mesh, spheres_noisy, view="front")
scan_evaluation_=evaluate_single_scan_from_landmarks(scan_name=scan_name,
                                    coords_pred=coords_pred,
                                    classes_pred=classes_pred,
                                    coords_gt=coords_gt,
                                    classes_gt=classes_gt,categories=CATEGORIES)
scan_evaluation_noisy=evaluate_single_scan_from_landmarks(scan_name=scan_name,
                                    coords_pred=shift_coords,
                                    classes_pred=shift_classes,
                                    coords_gt=coords_gt,
                                    classes_gt=classes_gt,categories=CATEGORIES)
scan_quality_profile=quality_profile_for_scan([scan_evaluation_],categories=CATEGORIES,scan_name=scan_name)
scan_quality_profile_noisy=quality_profile_for_scan([scan_evaluation_noisy],categories=CATEGORIES,scan_name=scan_name)



## num flip==> 2
### global shift with len(candidate)==> 86
## num shift==> 51


In [92]:
print('scan senza rumore==>',scan_quality_profile['mAP'], scan_quality_profile['quality_global'])
print('scan CON rumore==>',scan_quality_profile_noisy['mAP'], scan_quality_profile_noisy['quality_global'])


scan senza rumore==> 0.75 5
scan CON rumore==> 0.70 4


In [ ]:
fraction=0.5,
target_landmark_classes=['Mesial']
==> ## num flip==> 7

scan senza rumore==> 0.75 5
scan CON rumore==> 0.63 2
#####
fraction=0.3,
## num flip==> 4
scan senza rumore==> 0.75 5
scan CON rumore==> 0.67 3
#####
    fraction=0.2,
## num flip==> 2
scan senza rumore==> 0.75 5
scan CON rumore==> 0.70 4

In [59]:
classes_pred

['Mesial',
 'Mesial',
 'Mesial',
 'Mesial',
 'Mesial',
 'Mesial',
 'Mesial',
 'Mesial',
 'Mesial',
 'Mesial',
 'Mesial',
 'Mesial',
 'Mesial',
 'Mesial',
 'Distal',
 'Distal',
 'Distal',
 'Distal',
 'Distal',
 'Distal',
 'Distal',
 'Distal',
 'Distal',
 'Distal',
 'Distal',
 'Distal',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'Cusp',
 'InnerPoint',
 'InnerPoint',
 'InnerPoint',
 'InnerPoint',
 'InnerPoint',
 'InnerPoint',
 'InnerPoint',
 'InnerPoint',
 'InnerPoint',
 'InnerPoint',
 'OuterPoint',
 'OuterPoint',
 'OuterPoint',
 'OuterPoint',
 'OuterPoint',
 'OuterPoint',
 'OuterPoint',
 'OuterPoint',
 'OuterPoint',
 'OuterPoint',
 'OuterPoint',
 'OuterPoint',
 'FacialPoint',
 'FacialPoint',
 'FacialPoint',
 'FacialPoint',
 'FacialPoint',
 'FacialPoint',
 'FacialPoint',
 'FacialPoint',
 'FacialPoint

In [ ]:
scan_evaluation_.keys()

In [95]:
#Definizione pertubazioni da applicare 
PERTURBATIONS=[] 
#landmark mancanti
miss_landmark_configs_par={
"global_cfg":{'frac':[0.05,0.1]},
"local_cfg":{
    'cls':["Mesial",
            "Distal",
        "InnerPoint",
        "OuterPoint"], 
        'frac':[0.3,0.6]}}
#flip della classe di landmark 
flip_landmark_configs_par={
"global_cfg":{'frac':[0.05]},
"local_cfg":{
    'cls':["Mesial",
            "Distal",
        "InnerPoint",
        "OuterPoint"], 
        'frac':[0.2,0.3,0.5]}}
#shift classi di landmark
shift_landmark_configs_par={
    "global_cfg":{
            "frac": [0.20,0.40],
            "shift_mm": [0.5,1.0, 2.0, 3.0],
        },
    "local_cfg":{
            "cls": [
                "Mesial",
                "OuterPoint",
            ],
            "frac": [0.20,0.50],
            "shift_mm": [1.0, 2.0, 3.0],
        }}
#identifica con una label la configurazione scelta
miss_landmark_config_label = build_perturbation_label(miss_landmark_configs_par)
miss_landmark_configs_list= build_missing_landmark_configs(
    global_config=miss_landmark_configs_par['global_cfg'],
    local_config= miss_landmark_configs_par['local_cfg'])
class_flip_config_list=build_class_flip_configs(
    global_config=flip_landmark_configs_par['global_cfg'],
    local_config= flip_landmark_configs_par['local_cfg'])
shift_landmark_configs_list = build_landmark_shift_configs(
        global_config=shift_landmark_configs_par['global_cfg'],
        local_config= shift_landmark_configs_par['local_cfg'])
PERTURBATIONS.extend(shift_landmark_configs_list)

In [96]:
len(PERTURBATIONS)

20

In [97]:
PERTURBATIONS

[{'type': 'landmark_shift',
  'params': {'fraction': 0.2, 'shift_mm': 0.5, 'target_landmark_classes': []},
  'metadata': {'description': 'spostato 20% landmarks globalmente di 0.5mm'}},
 {'type': 'landmark_shift',
  'params': {'fraction': 0.2, 'shift_mm': 1.0, 'target_landmark_classes': []},
  'metadata': {'description': 'spostato 20% landmarks globalmente di 1.0mm'}},
 {'type': 'landmark_shift',
  'params': {'fraction': 0.2, 'shift_mm': 2.0, 'target_landmark_classes': []},
  'metadata': {'description': 'spostato 20% landmarks globalmente di 2.0mm'}},
 {'type': 'landmark_shift',
  'params': {'fraction': 0.2, 'shift_mm': 3.0, 'target_landmark_classes': []},
  'metadata': {'description': 'spostato 20% landmarks globalmente di 3.0mm'}},
 {'type': 'landmark_shift',
  'params': {'fraction': 0.4, 'shift_mm': 0.5, 'target_landmark_classes': []},
  'metadata': {'description': 'spostato 40% landmarks globalmente di 0.5mm'}},
 {'type': 'landmark_shift',
  'params': {'fraction': 0.4, 'shift_mm': 

In [221]:
miss_landmark_configs_list[0]['metadata']

{'description': 'rimosso 5% landmarks globalmente'}

In [67]:
scan_names=['F92OKIOI_upper', 'K4BAII5F_upper', 'RZ69JRI9_lower']
oracle_dataset=build_synthetic_oracle_dataset([scan_names[2]], SCANS, GT_ROOT, PRED_CSV,SCREENSHOT_DIR,pertubation_configs=PERTURBATIONS)

### applica flip
### applica flip
### applica flip
### applica flip
### applica flip
### applica flip
### applica flip
### applica flip
### applica flip
### applica flip
### applica flip
### applica flip
### applica flip


In [68]:
oracle_dataset[0].keys()

dict_keys(['scan', 'origin_pred_scan_conf_name', 'origin_pred_count_per_class', 'origin_pred_quality_score', 'origin_pred_gmap', 'origin_pred_quality_per_class', 'origin_pred_metadata', 'origin_pred_image_path', 'pred_scan_conf_name', 'pred_count_per_class', 'pred_quality_score', 'pred_gmap', 'pred_quality_per_class', 'pred_metadata', 'pred_image_path', 'mesh_path', 'gt_path', 'seg_path'])

In [69]:
round(float(oracle_dataset[0]['origin_pred_gmap']),3)
oracle_dataset[0]['pred_scan_conf_name']

'RZ69JRI9_lower_class_flip_target_landmark_classes___fraction_0.05'

In [208]:
for scan_ in oracle_dataset:
    print('#######')
    print('scan==>',scan_['scan'])

    print('pertubazione name=>',scan_['pred_scan_conf_name'])
    print('quality origin==>',scan_['origin_pred_quality_score'])
    print('quality pred==>',scan_['pred_quality_score'])
    print('maAP origin==>',scan_['origin_pred_gmap'])
    print('maAP pred==>',scan_['pred_gmap'])


#######
scan==> RZ69JRI9_lower
pertubazione name=> RZ69JRI9_lower_missing_landmarks_target_landmark_classes___fraction_0.05
quality origin==> 5
quality pred==> 4
maAP origin==> 0.7461829135494465
maAP pred==> 0.7051476661198882
#######
scan==> RZ69JRI9_lower
pertubazione name=> RZ69JRI9_lower_missing_landmarks_target_landmark_classes___fraction_0.1
quality origin==> 5
quality pred==> 3
maAP origin==> 0.7461829135494465
maAP pred==> 0.682666884919676
#######
scan==> RZ69JRI9_lower
pertubazione name=> RZ69JRI9_lower_missing_landmarks_fraction_0.3__target_landmark_classes_Mesial
quality origin==> 5
quality pred==> 4
maAP origin==> 0.7461829135494465
maAP pred==> 0.7083768153266816
#######
scan==> RZ69JRI9_lower
pertubazione name=> RZ69JRI9_lower_missing_landmarks_fraction_0.6__target_landmark_classes_Mesial
quality origin==> 5
quality pred==> 3
maAP origin==> 0.7461829135494465
maAP pred==> 0.6713309599474929
#######
scan==> RZ69JRI9_lower
pertubazione name=> RZ69JRI9_lower_missing_landma